In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go


In [39]:

# Create a 1-row, 3-column layout for subplots with shared axes
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=("Plot 1", "Plot 2", "Plot 3")
)

# Add traces to the first subplot (scatter plot)
fig.add_trace(
    go.Scatter(x=[1, 2, 3], y=[4, 5, 6], mode='lines', name='Line 1'),
    row=1, col=1
)

# Add traces to the second subplot (scatter plot)
fig.add_trace(
    go.Scatter(x=[1, 2, 3], y=[6, 5, 4], mode='lines', name='Line 2'),
    row=1, col=2
)

# Add traces to the third subplot (scatter plot)
fig.add_trace(
    go.Scatter(x=[1, 2, 3], y=[4, 6, 5], mode='lines', name='Line 3'),
    row=1, col=3
)

# Update layout to adjust spacing and labels
fig.update_layout(
    title_text="Horizontally Aligned Scatter Plots with Shared Axes",
    showlegend=True,  # Show the legend
    xaxis_title="X Axis",  # Set common X axis title
    yaxis_title="Y Axis"   # Set common Y axis title
)

# Show the figure
fig.show()


In [2]:
thresholds = ['0.1','0.5','0.8','A','I','N']
hardware = ["kyiv", "brisbane", "sherbrooke"]
mutant_types = ["equiv", "normal", "balanced"]
metrics_names = {'C':'chisquare', 'H':'hellinger', 'J':'jensenshannon', 'T':'trace', 'F':'fidelity', 'E':'expectation'}
#metrics = {'C':'Chisquare', 'H':'Hellinger', 'J':'Jensen-shannon', 'T':'Trace', 'F':'Fidelity', 'E':'Expectation Values'}
output_type = {'ae': 'Dominant', 'qpeexact': 'Dominant', 'vqe': 'Dominant', 'qft': 'Diverse', 'qftentangled': 'Diverse', 'wstate': 'Diverse'}


In [3]:
table_data = {
    "Output_type": ["Dominant", "Diverse"],
    "Input_type": ["PureState", "Quratest"],
    "Operator": ["Add", "Remove", "Replace"],
    "Gate_type": ["Single_qubit", "Multi_qubit"],
    "Relative_position": ["beginning", "pre_middle", "middle", "post_middle", "end"]
}


In [4]:
def getModelTolerance(model):
    if model == 'brisbane':
        tolerance_values_noisy = {
            'fidelity': 1 - 0.9818071588272935,
            'trace': 0.9474790361650993,
            'hellinger': 0.9001136659697,
            'jensenshannon': 0.7718372511093367,
            'chisquare': 8.822528185214266e-158,
            'expectation': 0.7070786758337857
        }
    elif model == 'sherbrooke':
        tolerance_values_noisy = {
            'fidelity': 1 - 0.9817918801809562,
            'trace': 0.9165467778554671,
            'hellinger': 0.8723835633308122,
            'jensenshannon': 0.7525100350370049,
            'chisquare': 3.1954543753995917e-141,
            'expectation': 0.5896550492107876
        }
    elif model == 'kyiv':
        tolerance_values_noisy = {
            'fidelity': 1 - 0.9817973573897236,
            'trace': 0.9134759188590066,
            'hellinger': 0.8760278316636461,
            'jensenshannon': 0.7549426398105366,
            'chisquare': 5.687357104528032e-162,
            'expectation': 0.6140893479852809
        }

    else:
        tolerance_values_noisy = {}

    return tolerance_values_noisy

# Get datasets

In [5]:
dtype_dict = {
    "gates": int,
    "depth": int,
    "singlequbit_gates": int,
    "multiqubit_gates": int,
    "Input": str,
    "Input_type": str,
    "Algorithm": str,
    "Qubits_number": int,
    "Operator": str,
    "Gate": str,
    "Position": int,
    "Qubits": int,
    "Gate_type": str,
    "Relative_position": str,
    "Output_type": str,
    "hardware": str,
    "threshold": str,
    "metric": str,
    "distance": float,
    "true_label": bool,
    "predicted_label": bool,
    "correctness": bool
}

csv_normal_path = f'results/results_normal_selected.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=dtype_dict)

csv_equiv_path = f'results/results_equiv_selected.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=dtype_dict)


In [6]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['true_label'] = np.where(df['true_label'] == True, "non-equivalent", "equivalent")
df['predicted_label'] = np.where(df['predicted_label'] == True, "non-equivalent", "equivalent")
print(df.head())

   gates  depth  singlequbit_gates  multiqubit_gates        Input Input_type  \
0     65     42                 29                36  PureState_0  PureState   
1     65     42                 29                36   Quratest_0   Quratest   
2     65     42                 29                36  PureState_1  PureState   
3     65     42                 29                36   Quratest_1   Quratest   
4     65     42                 29                36  PureState_2  PureState   

  Algorithm  Qubits_number Operator Gate  ...    Gate_type  Relative_position  \
0        ae              8      Add   ch  ...  Multi_qubit          beginning   
1        ae              8      Add   ch  ...  Multi_qubit          beginning   
2        ae              8      Add   ch  ...  Multi_qubit          beginning   
3        ae              8      Add   ch  ...  Multi_qubit          beginning   
4        ae              8      Add   ch  ...  Multi_qubit          beginning   

  Output_type hardware threshold

# Helper

In [7]:
# Helper function to setup layout and save image
def setup_layout_and_save(fig, title, folder_name, file_name, yaxis_range=None):
    fig.update_layout(
        title_text=title,
        height=400,
        width=2000,
        showlegend=True,
        yaxis_range=yaxis_range  # Set y-axis range if provided
    )
    os.makedirs(folder_name, exist_ok=True)
    fig.write_image(f"{folder_name}/{file_name}.png")# engine='orca')


# Get Box Plots

In [16]:
def print_box_plot(df, threshold_value, cat, file_name):

    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['true_label'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['true_label'] = df['true_label'].map(label_mapping)
    
    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(#scatter
        df, 
        y="distance", 
        x="x_offset", 
        color="true_label", 
        category_orders={cat: cat_range},
        points=False
    ) 
    
    # Add the threshold line
    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(cat_range) - 0.5,
        y0=threshold_value,
        y1=threshold_value,
        line=dict(color="red", dash="dash"),
        xref="x",
        yref="y",
    )
    
    # Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        #scattergap=0.75,
        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),
        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text="Expected value",
        boxgroupgap=0, 
        boxgap=0
    )
    
    # Save the figure
    setup_layout_and_save(fig, "RQ2", f'results/RQ2/', file_name, yaxis_range=[0, 1])


In [17]:
m = "F"
metric = "fidelity"
threshold = "0.1"
threshold_value = 0.1 #getModelTolerance(hw)[metric]
columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 

for hw in hardware:
    df_hw = df[df['hardware'] == hw]
    df_metric = df_hw[df_hw['metric'] == m]
    df_threshold = df_metric[df_metric['threshold'] == threshold]
    
    for key, categories in table_data.items():
        selected_columns = df_threshold[[key, 'true_label', 'distance']]
        file_name = f'{hw}_{threshold}_{key}'
        print_box_plot(selected_columns, threshold_value, key, file_name)
        
    for cat in columns: 
        min_val = int(df_equiv[cat].min())
        max_val = int(df_equiv[cat].max())
        cat_range = list(map(int, range(min_val, max_val + 1)))
        selected_columns = df_threshold[[cat, 'true_label', 'distance']]  
        file_name = f'{hw}_{threshold}_{cat}'
        print_box_plot(selected_columns, threshold_value, cat, file_name)

# Subplots

In [53]:
def print_scatter_plot(df, threshold_value, cat, file_name, subplot_idx, fig):
    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['true_label'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['true_label'] = df['true_label'].map(label_mapping)
    
    # Create the scatter plot with the adjusted x-axis values
    scatter = go.Scatter(
        y=df['distance'],
        x=df['x_offset'],
        mode='markers',
        marker=dict(color=df['true_label'].map({"Equivalent mutant": 'blue', "Non-Equivalent mutant": 'red'}), size=10),
        name=cat
    )
    
    #scatter = px.scatter(
    #    df, 
    #    y="distance", 
     #   x="x_offset", 
     #   color="true_label", 
     #   category_orders={cat: cat_range}
   # )
    
    # Add the threshold line
    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(cat_range) - 0.5,
        y0=threshold_value,
        y1=threshold_value,
        line=dict(color="red", dash="dash"),
        xref="x",
        yref="y",
    )
    
    # Add the scatter plot to the subplot
    fig.add_trace(scatter, row=1, col=subplot_idx)


In [54]:
def create_subplots_for_hardware(hw, threshold_value, table_data, df, m, threshold):
    # Create the subplot figure with enough subplots for each category
    fig = sp.make_subplots(
        rows=1,
        cols=len(table_data),  # Number of categories as columns
        shared_yaxes=True,  # Same y-axis for better comparison
        subplot_titles=list(table_data.keys()),  # Titles as the keys of table_data
        horizontal_spacing=0.1  # Adjust space between subplots
    )
    
    df_metric = df[df['metric'] == m]
    df_threshold = df_metric[df_metric['threshold'] == threshold]
    
    # Loop over each category in table_data and add it as a subplot
    for subplot_idx, (key, categories) in enumerate(table_data.items(), start=1):
        selected_columns = df_threshold[[key, 'true_label', 'distance']]
        file_name = f'{hw}_{threshold}_{key}'
        print_scatter_plot(selected_columns, threshold_value, key, file_name, subplot_idx, fig)    
        
    # Adjust layout for better visualization
    fig.update_layout(
        title_text=f"{hw} Performance Metrics",
        height=600,
        width=2000,
        showlegend=True,
        xaxis_title="Characteristic",
        yaxis_title="Distance between original and mutant"
    )
    
    # Save the figure for this hardware
    setup_layout_and_save(fig, f"{hw} Performance Metrics", 'results/RQ2/hardware', f"{hw}_{threshold}_metrics", yaxis_range=[0, 1])
    #setup_layout_and_save(fig, "RQ2", f'results/RQ2/', file_name, yaxis_range=[0, 1])


In [55]:
# Example of usage for each hardware
m = "F"
metric = "fidelity"
threshold = "0.1"
threshold_value = 0.1  # This can be adjusted based on your model

for hw in ["kyiv"]: #hardware:
    create_subplots_for_hardware(hw, threshold_value, table_data, df, m, threshold)

KeyboardInterrupt: 

In [ ]:
# Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        scattergap=0.75,

        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),

        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text="Expected value"
    )   